In [0]:
from pyspark.sql.functions import (
    col, current_date, lit
)

silver_customer_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/customers"
)

customers = (
    spark.read
    .format("delta")
    .load(silver_customer_path)
)

customer_scd2 = (
    customers
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
)

scd2_path = (
    "abfss://silver@ecommercenidhi.dfs.core.windows.net/customer_scd2"
)

customer_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .save(scd2_path)

display(customer_scd2)

In [0]:
from pyspark.sql.functions import when

current_customers = (
    spark.read
    .format("delta")
    .load(silver_customer_path)
)

changed_customer = (
    current_customers
    .withColumn(
        "city",
        when(col("customer_id") == "C0008", "Mumbai")
        .otherwise(col("city"))
    )
)

display(
    changed_customer
    .filter(col("customer_id") == "C0008")
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

target = DeltaTable.forPath(spark, scd2_path)

# Close the old version
(
    target.alias("target")
    .merge(
        changed_customer.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        AND target.city <> source.city
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_to": "current_date()",
            "is_current": "false"
        }
    )
    .execute()
)

In [0]:
new_versions = (
    changed_customer
    .filter(col("customer_id") == "C0008")
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .save(scd2_path)

In [0]:
final_scd2 = (
    spark.read
    .format("delta")
    .load(scd2_path)
)

display(
    final_scd2
    .filter(col("customer_id") == "C0008")
    .orderBy("effective_from")
)